I2 portable reader candidate. New outputs are isolated under runs/. Constructed examples demonstrate methods; fixed filing facts/figures are historical teaching inputs. This does not reproduce a market strategy.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch06_probability_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# 第 6 章：概率模拟

本 Notebook 只生成教学构造数据，不读取真实市场记录。固定种子用于复核模拟，不代表对市场参数的估计。核心实现和参数检查位于 code/ch06_probability.py。

In [ ]:
from pathlib import Path
import sys

candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    path for path in candidates
    if (path / "code" / "ch06_probability.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "code"))

import ch06_probability as probability

## 条件概率与理论矩

先算低基率信号的后验，再查看 Bernoulli 与二项分布的均值和方差。这些输入均为构造参数。

In [ ]:
posterior = probability.posterior_given_positive(
    prior_probability=0.005,
    sensitivity=0.95,
    false_positive_rate=0.05,
)
bernoulli_mean, bernoulli_variance = probability.bernoulli_moments(0.30)
binomial_mean, binomial_variance = probability.binomial_moments(20, 0.30)

print(f"constructed P(event | positive) = {posterior:.8%}")
print(f"Bernoulli(p=0.30): mean={bernoulli_mean:.4f}, variance={bernoulli_variance:.4f}")
print(f"Binomial(n=20,p=0.30): mean={binomial_mean:.4f}, variance={binomial_variance:.4f}")

## 固定种子的路径模拟

样本均值路径用 Bernoulli(0.30) 观测；CLT 图用多次独立重复计算标准化样本均值；随机游走和 Poisson 过程依各自定义生成增量并累加。

In [ ]:
sample_sizes, lln_paths = probability.simulate_lln_paths(
    probability=0.30, n_paths=24, n_steps=1000, seed=641
)
clt_values = probability.simulate_clt_standardized_means(
    probability=0.30, sample_size=120, replications=30_000, seed=642
)
walk_steps, walk_paths = probability.simulate_random_walk_paths(
    n_paths=18, n_steps=300, seed=643
)
poisson_times, poisson_counts = probability.simulate_poisson_process_paths(
    rate=0.70, horizon=12.0, n_intervals=240, n_paths=14, seed=644
)
print("LLN paths:", lln_paths.shape, "| final mean across paths:", round(float(lln_paths[:, -1].mean()), 4))
print("CLT replications:", clt_values.shape, "| mean / variance:", round(float(clt_values.mean()), 4), round(float(clt_values.var(ddof=1)), 4))
print("random walk paths:", walk_paths.shape, "| Poisson paths:", poisson_counts.shape)
print("Poisson terminal count mean:", round(float(poisson_counts[:, -1].mean()), 4))

## 图形

图的横纵轴、构造参数与理论参照均已在标题、图例或坐标标签中标注。

In [ ]:
from IPython.display import Image, display

figure_paths = probability.generate_figures(
    RUN_DIR / "figures" / "a2", seed=641
)
for name, path in figure_paths.items():
    print(f"{name}: {path.relative_to(PROJECT_ROOT)}")
    display(Image(filename=str(path)))

## 解释边界

LLN 与 CLT 图展示已知构造分布下的抽样行为；另外两图展示随机游走与独立 Poisson 增量构成的路径。它们不估计市场参数、不证明市场收益独立同分布，也不评价策略绩效。